In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("GoldLayerAnalysis") \
    .getOrCreate()

In [3]:
BASE_PATH = "/home/jovyan/work/output"

fact_order_items = spark.read.csv(
    f"{BASE_PATH}/gold/fact_order_items",
    header=True,
    inferSchema=True
)

dim_products = spark.read.csv(
    f"{BASE_PATH}/gold/dim_products",
    header=True,
    inferSchema=True
)

dim_sellers = spark.read.csv(
    f"{BASE_PATH}/gold/dim_sellers",
    header=True,
    inferSchema=True
)

In [4]:
fact_order_items.createOrReplaceTempView("fact_order_items")
dim_products.createOrReplaceTempView("dim_products")
dim_sellers.createOrReplaceTempView("dim_sellers")

In [ ]:
query = """
WITH seller_metrics AS (
    SELECT
        ds.seller_id,
        ds.seller_state,
        COUNT(*) AS total_orders,
        SUM(f.price + f.freight_value) AS total_revenue,

        -- Late delivery rate
        AVG(CASE WHEN CAST(f.is_late_delivery AS BOOLEAN) = true THEN 1 ELSE 0 END) AS late_delivery_rate,

        -- Avg delivery deviation
        AVG(f.days_delivery_vs_estimate) AS avg_days_vs_estimate

    FROM fact_order_items f
    JOIN dim_sellers ds
        ON f.seller_key = ds.seller_key
    GROUP BY ds.seller_id, ds.seller_state
),

filtered AS (
    SELECT *
    FROM seller_metrics
    WHERE total_orders >= 20
),

percentiles AS (
    SELECT
        *,
        -- Inverted (lower is better)
        1 - PERCENT_RANK() OVER (ORDER BY late_delivery_rate) AS on_time_pctl,
        1 - PERCENT_RANK() OVER (ORDER BY avg_days_vs_estimate) AS speed_pctl,

        -- Higher is better
        PERCENT_RANK() OVER (ORDER BY total_revenue) AS revenue_pctl
    FROM filtered
),

scored AS (
    SELECT
        *,
        (0.4 * on_time_pctl +
         0.3 * speed_pctl +
         0.3 * revenue_pctl) AS composite_score
    FROM percentiles
)

SELECT
    seller_id,
    seller_state,
    total_orders,
    total_revenue,
    late_delivery_rate,
    avg_days_vs_estimate,
    on_time_pctl,
    speed_pctl,
    revenue_pctl,
    composite_score,
    RANK() OVER (ORDER BY composite_score DESC) AS overall_rank
FROM scored
ORDER BY overall_rank
"""

spark.sql(query).show(50, False)